# Classification – Blood Group Detection (Multi-Feature Source)

Notebook này mở rộng từ notebook 03 gốc để hỗ trợ ba nguồn feature:

| Notebook nguồn | File CSV | Phương pháp |
|---|---|---|
| 02 | `processed/color_segmentation/color_segmentation_features.csv` | Color, LBP, GLCM, Gradient, Contour |
| 04 | `processed/frequency_domain/frequency_domain_features.csv` | FFT Ring Spectrum, DoG Wavelet |
| 05 | `processed/morphological_spatial/morphological_spatial_features.csv` | Morphological Profile, Hu Moments, Spatial |

**Chế độ feature** (chọn bằng `FEATURE_MODE`):
- `"color"` : chỉ dùng notebook 02
- `"frequency"` : chỉ dùng notebook 04
- `"morphological"` : chỉ dùng notebook 05
- `"color+frequency"` : merge 02 + 04
- `"color+morphological"` : merge 02 + 05
- `"all"` : merge cả ba

Mục tiêu: train nhiều mô hình, so sánh theo `f1_macro`, lưu model tốt nhất.


## 0. Config

In [ ]:
from pathlib import Path
from datetime import datetime
import json

import joblib, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

ROOT = Path.cwd()
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Paths to feature CSVs ─────────────────────────────────────────────────────
FEATURE_PATHS = {
    "color":         ROOT / "processed" / "color_segmentation"   / "color_segmentation_features.csv",
    "frequency":     ROOT / "processed" / "frequency_domain"     / "frequency_domain_features.csv",
    "morphological": ROOT / "processed" / "morphological_spatial"/ "morphological_spatial_features.csv",
}

# ── Main settings ─────────────────────────────────────────────────────────────
#  Options: "color", "frequency", "morphological",
#           "color+frequency", "color+morphological", "all"
FEATURE_MODE      = "all"

TARGET            = "blood_group"   # "blood_group" | "abo" | "rh"
USE_ORIGINAL_SPLIT= False
TEST_SIZE         = 0.20
RANDOM_STATE      = 42

ROOT, MODEL_DIR

## 1. Load & Merge Feature CSVs

In [ ]:
META_COLS = ["split", "file_name", "blood_group", "abo", "rh"]

def load_csv(key):
    path = FEATURE_PATHS[key]
    if not path.exists():
        raise FileNotFoundError(f"Feature CSV not found: {path}\nRun notebook {{'color':'02','frequency':'04','morphological':'05'}[key]} first.")
    df = pd.read_csv(path)
    print(f"  [{key}] {df.shape}")
    return df

def merge_feature_sources(mode: str) -> pd.DataFrame:
    """Load và merge các CSV theo mode. Join trên file_name."""
    keys = {
        "color":              ["color"],
        "frequency":          ["frequency"],
        "morphological":      ["morphological"],
        "color+frequency":    ["color", "frequency"],
        "color+morphological":["color", "morphological"],
        "all":                ["color", "frequency", "morphological"],
    }[mode]

    dfs = {}
    for k in keys:
        dfs[k] = load_csv(k)

    if len(dfs) == 1:
        return list(dfs.values())[0]

    # Merge on file_name; keep META_COLS from first df, suffix duplicates
    base_key = keys[0]
    merged   = dfs[base_key].copy()
    for k in keys[1:]:
        other = dfs[k]
        # Drop duplicate META_COLS from other (keep base)
        other_feat = other.drop(columns=[c for c in META_COLS if c in other.columns], errors="ignore")
        # Remove columns already in merged to avoid _x/_y suffix clutter
        other_feat = other_feat[[c for c in other_feat.columns if c not in merged.columns]]
        merged = pd.concat([merged, other_feat], axis=1)
    return merged

print(f"Feature mode: {FEATURE_MODE}")
df = merge_feature_sources(FEATURE_MODE)
print(f"Merged shape: {df.shape}")
print(f"NaN total: {int(df.isna().sum().sum())}")
df.head()

## 2. Prepare Features

In [ ]:
numeric_cols  = df.select_dtypes(include="number").columns.tolist()
feature_cols  = [c for c in numeric_cols if c not in META_COLS]

X = df[feature_cols]
y = df[TARGET].astype(str)

print(f"Target:        {TARGET}")
print(f"Feature count: {len(feature_cols)}")
print(f"Classes:       {sorted(y.unique())}")
print("\nTarget distribution:")
print(y.value_counts())

## 3. Train/Test Split

In [ ]:
if USE_ORIGINAL_SPLIT:
    train_mask = df["split"] == "train"
    test_mask  = df["split"].isin(["valid", "test"])
    X_train, X_test = X.loc[train_mask], X.loc[test_mask]
    y_train, y_test = y.loc[train_mask], y.loc[test_mask]
    print("Using original split.")
else:
    min_count = y.value_counts().min()
    if min_count < 2:
        raise ValueError("Stratified split requires at least 2 samples per class.")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
    print("Using stratified split.")

print(f"X_train: {X_train.shape}  X_test: {X_test.shape}")
print("\nTrain distribution:"); print(y_train.value_counts())
print("\nTest distribution:");  print(y_test.value_counts())

## 4. Define Candidate Models

Notebook này bổ sung **Gradient Boosting** so với notebook 03 gốc, phù hợp hơn khi feature space lớn từ ba nguồn.

| Model | Ghi chú |
|---|---|
| Logistic Regression | Baseline tuyến tính |
| Linear SVM | Tuyến tính mạnh khi nhiều feature |
| RBF SVM | Phi tuyến, kernel Gaussian |
| KNN | Khoảng cách trong feature space |
| Random Forest | Ensemble cây bất biến với scale |
| Extra Trees | Ngẫu nhiên hơn RF, thường nhanh hơn |
| Gradient Boosting | Boosting từng bước, thường mạnh nhất trên tabular |


In [ ]:
def scaled_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("scaler",  StandardScaler()),
        ("model",   model),
    ])

def tree_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("model",   model),
    ])

models = {
    "logistic_regression": scaled_pipeline(
        LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE)),
    "linear_svm": scaled_pipeline(
        LinearSVC(class_weight="balanced", max_iter=10000, random_state=RANDOM_STATE)),
    "rbf_svm": scaled_pipeline(
        SVC(kernel="rbf", C=10.0, gamma="scale", class_weight="balanced", random_state=RANDOM_STATE)),
    "knn": scaled_pipeline(
        KNeighborsClassifier(n_neighbors=5)),
    "random_forest": tree_pipeline(
        RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)),
    "extra_trees": tree_pipeline(
        ExtraTreesClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)),
    "gradient_boosting": tree_pipeline(
        GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=4,
                                   subsample=0.8, random_state=RANDOM_STATE)),
}

print("Models:", list(models.keys()))

## 5. Cross-Validation & Test Evaluation

In [ ]:
min_train_count = y_train.value_counts().min()
n_splits = max(2, min(3, int(min_train_count)))
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

rows, fitted_models = [], {}
for name, pipe in models.items():
    print(f"Training {name}...", end=" ", flush=True)
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rows.append({
        "model":              name,
        "feature_mode":       FEATURE_MODE,
        "n_features":         len(feature_cols),
        "cv_f1_macro_mean":   cv_scores.mean(),
        "cv_f1_macro_std":    cv_scores.std(),
        "test_accuracy":      accuracy_score(y_test, pred),
        "test_f1_macro":      f1_score(y_test, pred, average="macro"),
        "test_f1_weighted":   f1_score(y_test, pred, average="weighted"),
    })
    fitted_models[name] = pipe
    print(f"cv={cv_scores.mean():.3f}±{cv_scores.std():.3f}  "
          f"test_f1_macro={rows[-1]['test_f1_macro']:.3f}")

results_df = pd.DataFrame(rows).sort_values("test_f1_macro", ascending=False).reset_index(drop=True)
results_df

## 6. Best Model – Classification Report & Confusion Matrix

In [ ]:
best_name  = results_df.loc[0, "model"]
best_model = fitted_models[best_name]
best_pred  = best_model.predict(X_test)

print(f"Best model: {best_name}  (feature_mode={FEATURE_MODE})")
print(classification_report(y_test, best_pred, digits=4))

In [ ]:
labels = sorted(y.unique())
cm     = confusion_matrix(y_test, best_pred, labels=labels)

fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title(f"Confusion Matrix – {best_name} [{FEATURE_MODE}]")
plt.tight_layout()
plt.show()

## 7. Feature Importance (Tree-Based Models)

In [ ]:
if hasattr(best_model["model"], "feature_importances_"):
    imp = pd.Series(best_model["model"].feature_importances_, index=feature_cols)
    top20 = imp.nlargest(20)
    fig, ax = plt.subplots(figsize=(10, 6))
    top20.sort_values().plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title(f"Top-20 Feature Importances – {best_name} [{FEATURE_MODE}]")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.show()
    print("\nTop 20 features:")
    print(top20.round(5))
else:
    print(f"{best_name} does not support feature_importances_.")

## 8. Feature Mode Comparison (Optional)

In [ ]:
# Chạy cell này để so sánh các feature mode khác nhau mà không cần restart.
# Có thể mất vài phút nếu dataset lớn.

RUN_COMPARISON = False   # Đổi thành True để chạy

if RUN_COMPARISON:
    all_modes = ["color", "frequency", "morphological", "color+frequency", "color+morphological", "all"]
    compare_rows = []
    for mode in all_modes:
        try:
            df_m = merge_feature_sources(mode)
            num  = df_m.select_dtypes(include="number").columns
            fc   = [c for c in num if c not in META_COLS]
            X_m  = df_m[fc]; y_m = df_m[TARGET].astype(str)
            Xtr, Xte, ytr, yte = train_test_split(
                X_m, y_m, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_m)
            pipe = tree_pipeline(
                ExtraTreesClassifier(n_estimators=200, class_weight="balanced",
                                     n_jobs=-1, random_state=RANDOM_STATE))
            pipe.fit(Xtr, ytr)
            pred = pipe.predict(Xte)
            compare_rows.append({
                "feature_mode": mode,
                "n_features": len(fc),
                "test_f1_macro": f1_score(yte, pred, average="macro"),
                "test_accuracy": accuracy_score(yte, pred),
            })
            print(f"[{mode}] n_feat={len(fc)}  f1_macro={compare_rows[-1]['test_f1_macro']:.3f}")
        except Exception as e:
            print(f"[{mode}] SKIP: {e}")

    compare_df = pd.DataFrame(compare_rows).sort_values("test_f1_macro", ascending=False)
    display(compare_df)

    fig, ax = plt.subplots(figsize=(9, 4))
    compare_df.sort_values("test_f1_macro").plot(
        x="feature_mode", y="test_f1_macro", kind="barh", ax=ax, color="teal", legend=False)
    ax.set_xlabel("Test F1 Macro"); ax.set_title("Feature Mode Comparison (Extra Trees)")
    plt.tight_layout(); plt.show()
else:
    print("Set RUN_COMPARISON = True to run cross-mode comparison.")

## 9. Save Best Model

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
safe_mode = FEATURE_MODE.replace("+", "_")

joblib_path = MODEL_DIR / f"{safe_mode}_{TARGET}_{best_name}_{timestamp}.joblib"
pkl_path    = MODEL_DIR / f"{safe_mode}_{TARGET}_{best_name}_{timestamp}.pkl"
metrics_path= MODEL_DIR / f"{safe_mode}_{TARGET}_metrics_{timestamp}.csv"
report_path = MODEL_DIR / f"{safe_mode}_{TARGET}_classification_report_{timestamp}.txt"

model_package = {
    "model":              best_model,
    "model_name":         best_name,
    "feature_mode":       FEATURE_MODE,
    "target":             TARGET,
    "feature_columns":    feature_cols,
    "classes":            labels,
    "use_original_split": USE_ORIGINAL_SPLIT,
    "test_size":          TEST_SIZE,
    "random_state":       RANDOM_STATE,
    "saved_at":           timestamp,
    "metrics":            results_df.to_dict(orient="records"),
}

joblib.dump(model_package, joblib_path, compress=3)
with open(pkl_path, "wb") as f:
    pickle.dump(model_package, f)

results_df.to_csv(metrics_path, index=False)
with report_path.open("w", encoding="utf-8") as f:
    f.write(f"Best model: {best_name}\n")
    f.write(f"Feature mode: {FEATURE_MODE}\n")
    f.write(f"Target: {TARGET}\n")
    f.write(f"Saved at: {timestamp}\n\n")
    f.write(classification_report(y_test, best_pred, digits=4))

print(f"Saved model: {joblib_path}")
print(f"Saved pkl:   {pkl_path}")
print(f"Metrics:     {metrics_path}")
print(f"Report:      {report_path}")

## 10. Verify: Load & Predict

In [ ]:
loaded   = joblib.load(joblib_path)
loaded_m = loaded["model"]
loaded_f = loaded["feature_columns"]

sample    = df.sample(10, random_state=RANDOM_STATE)
# Chỉ dùng các cột feature_columns; reindex để đảm bảo thứ tự
sample_X  = sample[loaded_f] if all(c in sample.columns for c in loaded_f) else \
            pd.DataFrame(sample).reindex(columns=loaded_f, fill_value=0.0)
sample_pred = loaded_m.predict(sample_X)

pd.DataFrame({
    "file_name":  sample["file_name"].values,
    "true_label": sample[TARGET].values,
    "pred_label": sample_pred,
})

## 11. Ghi chú

**Điểm mới so với notebook 03 gốc:**
- Hỗ trợ 3 nguồn feature và 6 chế độ merge.
- Bổ sung model **Gradient Boosting** (thường mạnh nhất trên tabular data khi feature nhiều).
- Lưu thêm thông tin `feature_mode` vào model package và tên file.
- Cell so sánh feature mode (Section 8) có thể chạy độc lập mà không cần restart.
- Model package ghi `feature_columns` đầy đủ, đảm bảo inference app dùng đúng bộ feature.

**Gợi ý cho báo cáo:**
- Chạy `FEATURE_MODE = "all"` và `RUN_COMPARISON = True` để có bảng so sánh đầy đủ.
- So sánh macro F1 theo feature mode → thấy được bộ feature nào đóng góp nhiều nhất.
- Feature importance (Section 7) cho biết feature nào từ notebook 04/05 được Extra Trees xem trọng.
